In [ ]:
import numpy as np
from sklearn.utils import class_weight
import h5py

import tensorflow as tf
tf.keras.utils.set_random_seed(424)

import warnings
warnings.filterwarnings("ignore")

In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if len(gpus):
    # 设置 GPU 显存占用为按需分配，增长式
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
        	# 异常处理
        	print(e)

In [ ]:
# all subjects ECG data and ann data
with h5py.File(f"ppg_sig_20s_5s.h5", "r") as rf :
    ppg_sig = rf['ppg_sig'][:][:,::,:]
with h5py.File(f"lab_mtx_20s_5s.h5", "r") as rf :
    ann_seg = rf['lab_mtx'][:]

In [4]:
# V3, V2, A3, A2,  VA4, VA5
# 0,  1,  2,  3,   4,   5

lab_seg = np.zeros((len(ann_seg[:,0]),6))
for line in np.arange(len(ann_seg[:,0])) :
    # ====== V temp =======
    if ann_seg[line,0] <= 3.5 :
        lab_seg[line,0] = 0
        lab_seg[line,1] = 0
    elif ann_seg[line,0] <= 6.5 :
        lab_seg[line,0] = 1
        if ann_seg[line,0] <= 5 :
            lab_seg[line,1] = 0
        else:
            lab_seg[line,1] = 1
    else :
        lab_seg[line,0] = 2
        lab_seg[line,1] = 1
    # ====== A temp =======    
    if ann_seg[line,1] <= 3.5 :
        lab_seg[line,2] = 0
        lab_seg[line,3] = 0
    elif ann_seg[line,1] <= 6.5 :
        lab_seg[line,2] = 1
        if ann_seg[line,1] <= 5 :
            lab_seg[line,3] = 0
        else:
            lab_seg[line,3] = 1
    else :
        lab_seg[line,2] = 2
        lab_seg[line,3] = 1

    # ======= VA-4 temp ======
    if lab_seg[line,1] == 0 and lab_seg[line,3] == 0 :
        lab_seg[line,4] = 1
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 1 :
        lab_seg[line,4] = 0
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 0 :
        lab_seg[line,4] = 2
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 1 :
        lab_seg[line,4] = 3

    # ======== VA-5 temp ======
    if lab_seg[line,0] == 1 and lab_seg[line,2] == 1 :
        lab_seg[line,5] = 2
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 0 :
        lab_seg[line,5] = 1
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 1 :
        lab_seg[line,5] = 0
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 0 :
        lab_seg[line,5] = 3
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 1 :
        lab_seg[line,5] = 4

In [5]:
class FFTLayer1d(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FFTLayer1d, self).__init__(**kwargs)

    def call(self, inputs):
        # 对每个 channel 进行 FFT 变换
        fft_result1 = tf.signal.fft(tf.cast(inputs[:,:,0], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        amp_norm = np.ones((1,1280))/640
        amp_norm[:,0] = 0
        real_part1 = tf.math.real(fft_result1)*amp_norm
        imag_part1 = tf.math.imag(fft_result1)*amp_norm
        
        return tf.concat([real_part1[:,:,tf.newaxis], imag_part1[:,:,tf.newaxis]], axis=-1)

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, 2 * channel)
        return (input_shape[0], input_shape[1], 2 * input_shape[2])

In [6]:
class MtsclEmb(tf.keras.layers.Layer):
    def __init__(self, resnum, **kwargs):
        super(MtsclEmb, self).__init__(**kwargs)

        self.scl1_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=1,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl1_emb{resnum}'
        )
    
        self.scl2_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=2,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl2_emb{resnum}'
        )

        self.scl3_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=3,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl3_emb{resnum}'
        )
      
        self.scl4_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=4,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl4_emb{resnum}'
        )

        self.scl5_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=5,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl5_emb{resnum}'
        )
    
        self.scl6_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=6,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl6_emb{resnum}'
        )

        self.scl7_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=7,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl7_emb{resnum}'
        )
      
        self.scl8_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=8,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl8_emb{resnum}'
        )

        self.mscc = tf.keras.layers.Concatenate(name=f'mscc_emb{resnum}')
    
    def call(self, x):
        scl1 = self.scl1_cov(x)
        scl2 = self.scl2_cov(x)
        scl3 = self.scl3_cov(x)
        scl4 = self.scl4_cov(x)
        scl5 = self.scl5_cov(x)
        scl6 = self.scl6_cov(x)
        scl7 = self.scl7_cov(x)
        scl8 = self.scl8_cov(x)

        mscc = self.mscc([scl1,scl2,scl3,scl4,scl5,scl6,scl7,scl8])

        return mscc

    def compute_output_spec(self, input_spec):
        # 返回输出的 shape 和 dtype
        return tf.keras.KerasTensor(
            shape=(input_spec.shape[0], input_spec.shape[1], 16),
            dtype=input_spec.dtype
        )
    

In [7]:
class TransformerBlk(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, ffn_dim, **kwargs):
        super(TransformerBlk, self).__init__( **kwargs)
        self.mhatt = tf.keras.layers.MultiHeadAttention(num_heads=num_heads,key_dim=key_dim,dropout=0.1,kernel_initializer="he_normal",name="mhatt")
        self.add_att = tf.keras.layers.Add(name="add_att")
        self.ln_att = tf.keras.layers.LayerNormalization(name='ln_att')
        self.ffn = tf.keras.layers.Dense(units=ffn_dim,activation='gelu',kernel_initializer='he_normal',name='ffn_1')
        self.add_ffn = tf.keras.layers.Add(name="add_ffn")
        self.ln_ffn = tf.keras.layers.LayerNormalization(name='ln_ffn')

    def call(self, x) :
        mhatt = self.mhatt(x,x)
        add_att = self.add_att([x,mhatt])
        ln_att = self.ln_att(add_att)
        ffn = self.ffn(ln_att)
        add_ffn = self.add_ffn([ln_att,ffn])
        ln_ffn = self.ln_ffn(add_ffn)

        return ln_ffn

In [8]:
def EmoStumd(bs=32, fs: int=64, emo_num=3, distill=False) -> tf.keras.Model:
    # =========================================== signal layers resUnet ===============================================
    # =========================================== FFT layers resUnet ==================================================
    ipl = tf.keras.Input((20*fs,1),batch_size=bs)
    FFT_layer = FFTLayer1d()(ipl)
    # =========================================== signal Encoder ======================================================
    scl_sig = MtsclEmb(1)(ipl)

    # ============================================ FFT Encoder =======================================================
    scl_frq = MtsclEmb(2)(FFT_layer)

    tfb = TransformerBlk(num_heads=4,key_dim=16,ffn_dim=16)
    
    sig_tfb = tfb(scl_sig)
    frq_tfb = tfb(scl_frq)

    sig_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='sig_deco_gap')(sig_tfb)
    sig_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='sig_deco_gmp')(sig_tfb)
    sig_deco_cc = tf.keras.layers.concatenate([sig_deco_gap,sig_deco_gmp],name="sig_deco_cc")
    sig_dec_f = tf.keras.layers.Dense(units=16,name="sig_dec_f")(sig_deco_cc)
    sig_dec_gelu = tf.keras.layers.Activation(activation='tanh',name="sig_dec_gelu")(sig_dec_f)

    frq_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='frq_deco_gap')(frq_tfb)
    frq_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='frq_deco_gmp')(frq_tfb)
    frq_deco_cc = tf.keras.layers.concatenate([frq_deco_gap,frq_deco_gmp],name="frq_deco_cc")
    frq_dec_f = tf.keras.layers.Dense(units=16,name="frq_dec_f")(frq_deco_cc)
    frq_dec_gelu = tf.keras.layers.Activation(activation='tanh',name="frq_dec_gelu")(frq_dec_f)
    
    ft_cc = tf.keras.layers.concatenate([sig_deco_cc,frq_deco_cc],name="ft_cc")
    ft_fc = tf.keras.layers.Dense(units=16,activation='tanh',name="ft_fc")(ft_cc)

    emo_add = tf.keras.layers.Add(name='emo_add')([sig_dec_gelu,frq_dec_gelu,ft_fc])
    emo_cls = tf.keras.layers.Dense(units=emo_num,activation='softmax',name='emo_cls')(emo_add)

    if distill :
        runet_emostu = tf.keras.Model(inputs=ipl,outputs=[emo_cls,sig_dec_gelu,frq_dec_gelu,ft_fc],name='runet_emostu')
    else:
        runet_emostu = tf.keras.Model(inputs=ipl,outputs=emo_cls,name='runet_emostu')
        
    return runet_emostu

In [ ]:
models_v = []
for sub in np.arange(30) :
    sub_model = EmoStumd()
    opt_stu = tf.keras.optimizers.AdamW(learning_rate=1e-3, clipnorm=1.)
    sub_model.compile(optimizer=opt_stu,loss='categorical_crossentropy')
    sub_model.load_weights(f'mout_distillation_res/case_distill/avg_v3_a3_ftsh/case_ptp_cls_loso_train_v3_s{sub}.valbest.keras')
    models_v.append(sub_model)

In [ ]:
tt_prob_v = np.zeros((len(ppg_sig[:,0,0]),3,30))
for sub in np.arange(30) :
    submd_prob_v = models_v[sub].predict(x=ppg_sig,verbose=0)
    tt_prob_v[:,:,sub] = submd_prob_v
m_prob_v = np.mean(tt_prob_v,axis=-1)

# No calibration

In [ ]:
allv_onehot_lab = tf.keras.utils.to_categorical(lab_seg[:,0], num_classes=3)

cea = tf.keras.metrics.CategoricalAccuracy()
mf1 = tf.keras.metrics.F1Score(average="macro")
wf1 = tf.keras.metrics.F1Score(average="weighted")

cea.update_state(allv_onehot_lab,m_prob_v)
wf1.update_state(allv_onehot_lab,m_prob_v)
mf1.update_state(allv_onehot_lab,m_prob_v)

print(f'Acc = {cea.result().numpy():.4f}, W-F1={mf1.result().numpy():.4f}, M-F1={wf1.result().numpy():.4f}')

In [12]:
def stack_md(cls=3) ->  tf.keras.Model:
    inputlayer = tf.keras.Input((cls,30))
    flatten = tf.keras.layers.Flatten()(inputlayer)
    fc_hid = tf.keras.layers.Dense(units=cls*15,activation='gelu',kernel_initializer='he_normal')(flatten)
    fc_out = tf.keras.layers.Dense(units=cls,activation='softmax')(fc_hid)

    embed_md = tf.keras.Model(inputs=inputlayer,outputs=fc_out,name='embed_md')

    return embed_md

# With Calibration

In [ ]:
X_v = tt_prob_v
Y_onehot_v = allv_onehot_lab
lr = 5e-2
file_path = '/case/'
for sub in np.arange(35) :
    use_idx = ann_seg[:,-1] == (sub+0)
    sub_x = X_v[use_idx,:,:]
    sub_y = Y_onehot_v[use_idx,:]

    Xtrain,Xtest,Ytrain,Ytest = train_test_split(sub_x,sub_y,test_size=0.5,random_state=42)
    spw = class_weight.compute_sample_weight(class_weight='balanced', y=Ytrain[:,0])
    
    embed = stack_md()
    opt_emb = tf.keras.optimizers.AdamW(learning_rate=lr, clipnorm=1.)
    embed.compile(optimizer=opt_emb, loss='categorical_crossentropy', metrics=["categorical_accuracy",tf.keras.metrics.F1Score(average="macro"),tf.keras.metrics.F1Score(average="weighted")])
    ckpt_ptp = tf.keras.callbacks.ModelCheckpoint(file_path+f'labdata_v3_s{sub}.valbest.keras', monitor='val_categorical_accuracy', verbose=0, save_best_only=True, mode='auto')
    _ = embed.fit(x=Xtrain, y=Ytrain, batch_size=32, epochs=100, verbose=0, sample_weight=spw, validation_data=(Xtest,Ytest), callbacks=[ckpt_ptp])
    
    embed.load_weights(file_path+f'labdata_v3_s{sub}.valbest.keras')
    loss,emo_acc,emo_f1,_,emo_f1w = embed.evaluate(x=Xtest,y=Ytest,verbose=0)
    print(f"Testing for Subject {sub+1} ... ACC={round(emo_acc,4)}, F1={round(emo_f1,4)}, F1W={round(emo_f1w,4)}")